# GCSE-level school quality: a general effect and its consistency

The subject notebooks each regress A-level VA on a single GCSE element. But a school is not judged on one GCSE element: an "outstanding" school is one whose GCSE value added is **consistently high across the board**. This notebook builds that picture at GCSE level, before it is linked to A-level.

For each school we have six GCSE value-added elements, each with a published confidence interval: English, Maths, Science, Humanities, Languages and Open. We model them together with two school-level latent quantities:

- a **general GCSE quality** $g_i$: how high the school's value added is across all elements;
- a **consistency** $s_i$: how much its elements scatter around that general level (small $s_i$ means consistent, large means uneven).

Each element is measured with known error, exactly as in the A-level notebooks. We also ask whether the two are related: are the schools with the highest general quality also the most consistent?

The data contain overall Progress 8 (`P8MEA`) and an EBacc element (`P8MEAEBAC`). We do not use them. `P8MEA` is almost exactly a weighted sum of English, Maths, EBacc and Open, and the EBacc element already contains Science, Humanities and Languages, so including them would count the same information twice.

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import pytensor.tensor as pt
import scipy.sparse as sp

%config InlineBackend.figure_format = 'retina'
RANDOM_SEED = 8927
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-darkgrid")

print(f"Running on PyMC v{pm.__version__}")

## Data

We reshape `all-value-add-errors.csv` to long format: one row per school × GCSE element with the published value added and its confidence interval. The standard error is `(upper - lower) / (2 * 1.96)`. English, Maths and Open are available for every school with Progress 8; Science, Humanities and Languages are missing for a few schools, which simply contribute fewer rows.

In [ ]:
raw = pd.read_csv("data/all-value-add-errors.csv")

elements = ["English", "Maths", "Science", "Humanities", "Languages", "Open"]
column = {"English": "P8MEAENG", "Maths": "P8MEAMAT", "Science": "SCIVAMEA_PTQ_EE",
          "Humanities": "HUMVAMEA_PTQ_EE", "Languages": "LANVAMEA_PTQ_EE", "Open": "P8MEAOPEN"}
# Number of pupils behind each element: Science, Humanities and Languages have their own; the Progress 8 elements use P8 pupils.
pupils_column = {e: (f"{column[e]} pupils" if f"{column[e]} pupils" in raw else "P8 pupils") for e in elements}

Z_95 = 1.96
frames = []
for e in elements:
    c = column[e]
    sub = raw[["URN", c, f"{c} lower", f"{c} upper", pupils_column[e]]].dropna()
    sub.columns = ["URN", "va", "lower", "upper", "pupils"]
    sub["element"] = e
    frames.append(sub)
long = pd.concat(frames)
long["se"] = (long["upper"] - long["lower"]) / (2 * Z_95)

# Keep schools with the three Progress 8 elements (all of them have these) so every school has at least three scores.
n_elements_per_school = long.groupby("URN")["element"].nunique()
keep = n_elements_per_school[n_elements_per_school >= 3].index
long = long[long["URN"].isin(keep)].sort_values("URN").reset_index(drop=True)

urns = pd.Index(sorted(long["URN"].unique()))
long["school_idx"] = urns.get_indexer(long["URN"])
long["element_idx"] = long["element"].map({e: k for k, e in enumerate(elements)}).to_numpy()

print(f"{len(urns)} schools, {len(long)} school-element observations")
print("elements per school:", long.groupby("URN")["element"].nunique().value_counts().sort_index().to_dict())
summary = long.groupby("element")[["va", "se"]].agg(["count", "mean", "std", "median"]).round(3).loc[elements]
summary

### Checking the confidence-interval formula

DfE's interval is $\pm 1.96\,\sigma_{national}/\sqrt{n}$, so $se\sqrt{n}$ should be constant within each element.

In [ ]:
print((long["se"] * np.sqrt(long["pupils"])).groupby(long["element"]).agg(["mean", "std"]).round(3).loc[elements])

Each element has its own near-constant national SD, so the standard errors are pure sampling noise. Languages has the largest standard error (about 0.21 against 0.11-0.14 for the others) because it counts only the pupils entered for languages.

## Exploratory look

Left: the raw correlation between elements across schools. Right: the eigenvalues of that correlation matrix, for schools with all six elements. A single dominant eigenvalue means one general factor accounts for most of the shared variation.

In [ ]:
wide = long.pivot(index="URN", columns="element", values="va")[elements]
complete = wide.dropna()
raw_corr = complete.corr()
eigvals = np.sort(np.linalg.eigvalsh(raw_corr.to_numpy()))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
axes[0].imshow(raw_corr, vmin=0.3, vmax=1, cmap="Blues")
axes[0].set_xticks(range(6), elements, rotation=30)
axes[0].set_yticks(range(6), elements)
for i in range(6):
    for j in range(6):
        axes[0].text(j, i, f"{raw_corr.iloc[i, j]:.2f}", ha="center", va="center")
axes[0].set_title(f"Raw correlation of GCSE VA elements ({len(complete)} schools with all six)")
axes[0].grid(False)
axes[1].bar(range(1, 7), eigvals / eigvals.sum(), color="#4C72B0")
axes[1].set_xlabel("component")
axes[1].set_ylabel("share of variance")
axes[1].set_title("Eigenvalues of the correlation matrix")
plt.tight_layout()
plt.show()
print("share of variance in first component:", round(eigvals[0] / eigvals.sum(), 3))

**One general factor dominates.** The first component carries 76% of the variance, and the second only 11%. English, Maths, Science, Humanities and Open all correlate 0.77-0.87 with each other, so a school that adds value in one is very likely to add it in the others. Languages is the outlier: it correlates only 0.44-0.49 with everything, so it is a much weaker indicator of the general level (it covers only pupils entered for languages and is measured with the largest error).

Two pairs stand out slightly above the rest: Maths-Science (0.87) and English-Open (0.86). Keep these in mind; the residual check near the end returns to them.

## Model

For school $i$ and element $e$ (English, Maths, Science, Humanities, Languages, Open), the observed value added $x^{obs}_{ie}$ has known standard error $\sigma_{ie}$:

$$
\begin{aligned}
x^{obs}_{ie} &\sim \text{Normal}(x_{ie}, \sigma_{ie}), \qquad x_{ie} = \mu_e + \lambda_e\, g_i + \delta_{ie}, \qquad \delta_{ie} \sim \text{Normal}(0,\ \tau_e\, s_i) \\
\log s_i &= \sigma_s\left(\rho\, g_i + \sqrt{1-\rho^2}\; w_i\right), \qquad g_i,\, w_i \sim \text{Normal}(0, 1)
\end{aligned}
$$

- $g_i$ is the school's **general quality**, on a scale of one standard deviation across schools. $\lambda_e > 0$ says how strongly element $e$ follows it and $\mu_e$ is the element's average.
- $\delta_{ie}$ is the school's **element-specific departure** from its general level. Its typical size is $\tau_e$ for an average school, multiplied by the school's consistency factor $s_i$. The prior on $\log s_i$ is centred at zero, so $s_i = 1$ is a typical school, $s_i = 2$ has twice the usual scatter and $s_i = 0.5$ half.
- $\sigma_s$ is how much schools differ in consistency. If $\sigma_s = 0$ every school is equally consistent and the model reduces to a plain one-factor model.
- $\rho$ is the correlation between general quality and $\log s_i$. $\rho < 0$ would mean the higher-quality schools are the more consistent ones.

As in the A-level factor models, we **marginalise out** $\delta_{ie}$ analytically: given $g_i$ and $s_i$, $x^{obs}_{ie} \sim \text{Normal}(\mu_e + \lambda_e g_i,\ \sqrt{\tau_e^2 s_i^2 + \sigma_{ie}^2})$. That removes a latent parameter per observation. We fit two versions: one with **constant** consistency ($s_i = 1$ for all schools) and the full model, to see whether allowing schools to differ in consistency is supported.

In [ ]:
n_schools = len(urns)
n_elements = len(elements)
x_obs = long["va"].to_numpy()
x_se = long["se"].to_numpy()
s_idx = long["school_idx"].to_numpy()
e_idx = long["element_idx"].to_numpy()

def build_model(vary_consistency):
    with pm.Model(coords={"element": elements}) as model:
        mu = pm.Normal("mu", 0, 1, dims="element")
        lam = pm.HalfNormal("lam", 1, dims="element")
        tau = pm.HalfNormal("tau", 0.5, dims="element")
        g = pm.Normal("g", 0, 1, shape=n_schools)
        if vary_consistency:
            sigma_s = pm.HalfNormal("sigma_s", 0.5)
            rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
            w = pm.Normal("w", 0, 1, shape=n_schools)
            log_s = pm.Deterministic("log_s", sigma_s * (rho * g + pt.sqrt(1 - rho**2) * w))
            specific_sd = tau[e_idx] * pt.exp(log_s[s_idx])
        else:
            specific_sd = tau[e_idx]
        pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx],
                  sigma=pt.sqrt(specific_sd**2 + x_se**2), observed=x_obs)
    return model

constant_model = build_model(vary_consistency=False)
full_model = build_model(vary_consistency=True)

### Prior predictive check

In [ ]:
with full_model:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RANDOM_SEED)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(prior.prior_predictive["x_obs"].to_numpy().ravel(), bins=60, color="#4C72B0", range=(-6, 6))
ax.set_title("Prior predictive: GCSE element VA")
plt.tight_layout()
plt.show()
print("observed VA range:", x_obs.min().round(2), "to", x_obs.max().round(2))
del prior

### Fit

In [ ]:
with constant_model:
    idata_const = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                            random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
with full_model:
    idata_full = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                           random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics

In [ ]:
for name, idata in [("constant consistency", idata_const), ("varying consistency", idata_full)]:
    print(f"{name}: divergences = {int(idata.sample_stats['diverging'].sum())}")
az.summary(idata_full, var_names=["mu", "lam", "tau", "sigma_s", "rho"], round_to=3)

## What the general factor shows

### How strongly does each element follow general quality?

$\lambda_e$ is the change in element $e$'s value added for a one-SD-higher $g_i$. The share of an element's between-school variance that is general is $\lambda_e^2 / (\lambda_e^2 + \tau_e^2 + \bar\sigma_e^2)$, where the last term is the typical sampling variance. The remainder is either element-specific departures from general quality or sampling noise.

In [ ]:
post = idata_full.posterior
pc = idata_const.posterior

def summarise(x):
    lo, hi = np.percentile(x, [5.5, 94.5])
    return f"mean={x.mean():.3f}, 89% interval [{lo:.3f}, {hi:.3f}]"

mean_se2 = long.groupby("element")["se"].apply(lambda s: (s**2).mean()).reindex(elements).to_numpy()
rows = []
for k, e in enumerate(elements):
    lam_k = post["lam"].sel(element=e).to_numpy().ravel()
    tau_k = post["tau"].sel(element=e).to_numpy().ravel()
    share_general = lam_k**2 / (lam_k**2 + tau_k**2 + mean_se2[k])
    share_noise = mean_se2[k] / (lam_k**2 + tau_k**2 + mean_se2[k])
    rows.append({"element": e, "loading_lam": lam_k.mean(), "specific_sd_tau": tau_k.mean(),
                 "typical_se": np.sqrt(mean_se2[k]), "share_general": share_general.mean(),
                 "share_specific": 1 - share_general.mean() - share_noise.mean(), "share_noise": share_noise.mean()})
loadings = pd.DataFrame(rows).set_index("element").round(3)
display(loadings)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for k, e in enumerate(elements):
    for ax, var, label in [(axes[0], "lam", "loading on general quality, $\\lambda_e$"), (axes[1], "tau", "typical element-specific SD, $\\tau_e$")]:
        lo, m, hi = np.percentile(post[var].sel(element=e).to_numpy().ravel(), [5.5, 50, 94.5])
        ax.plot([lo, hi], [k, k], color="#4C72B0", linewidth=2)
        ax.plot(m, k, "o", color="#4C72B0")
        ax.set_xlabel(label)
for ax in axes:
    ax.set_yticks(range(6), elements)
    ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Do schools really differ in consistency?

The full model lets each school have its own consistency $s_i$. If schools were all equally consistent, $\sigma_s$ would be near zero. Below: the posterior of $\sigma_s$ and of $\rho$ (the link between general quality and consistency).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(post["sigma_s"].to_numpy().ravel(), bins=40, density=True, color="#4C72B0")
axes[0].set_xlabel("sigma_s: spread of log consistency across schools")
axes[1].hist(post["rho"].to_numpy().ravel(), bins=40, density=True, color="#DD8452")
axes[1].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("rho: correlation of general quality with log s")
plt.tight_layout()
plt.show()
print("sigma_s:", summarise(post["sigma_s"].to_numpy().ravel()))
print("rho:    ", summarise(post["rho"].to_numpy().ravel()))
sig = post["sigma_s"].to_numpy().ravel()
print(f"a school one SD above/below average in log consistency has element-specific scatter x{np.exp(sig.mean()):.2f} / x{np.exp(-sig.mean()):.2f} the typical school's")

### A posterior predictive check: does constant consistency fit?

For each school and posterior draw we compute how surprising its six (or fewer) element scores are relative to the model, $T_i = \frac{1}{k_i}\sum_e r_{ie}^2$ with $r_{ie} = (x^{obs}_{ie} - \mu_e - \lambda_e g_i)/\sqrt{\tau_e^2 s_i^2 + \sigma_{ie}^2}$. Data that fit the model give $T_i$ near 1, and if some schools are more scattered than the model allows we see too many large $T_i$. We compare the observed share of schools above a few thresholds with the share in data replicated from the model, for the constant-consistency model and the full model.

In [ ]:
counts = np.bincount(s_idx, minlength=n_schools).astype(float)
school_matrix = sp.csr_matrix((np.ones(len(s_idx)), (s_idx, np.arange(len(s_idx)))), shape=(n_schools, len(s_idx)))
thin = slice(0, 1000, 5)   # 200 draws per chain, 800 in total

def draw_array(p, name, extra_dims=1):
    arr = p[name].to_numpy()[:, thin]
    return arr.reshape(-1, *arr.shape[2:])

def ppc_share(p, vary, thresholds=(1.5, 2.0, 3.0)):
    mu_d, lam_d, tau_d, g_d = (draw_array(p, n) for n in ["mu", "lam", "tau", "g"])
    n_draws = len(mu_d)
    log_s_d = draw_array(p, "log_s") if vary else np.zeros((n_draws, n_schools))
    mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
    sd = np.sqrt((tau_d[:, e_idx] * np.exp(log_s_d[:, s_idx])) ** 2 + x_se**2)
    rep = mean + sd * rng.standard_normal(mean.shape)
    out = {}
    for label, values in [("observed", x_obs[None, :]), ("replicated", rep)]:
        r2 = ((values - mean) / sd) ** 2
        T = (school_matrix @ r2.T).T / counts
        out[label] = np.stack([(T > t).mean(axis=1) for t in thresholds], axis=1)
    return out

rows = []
for name, p, vary in [("constant consistency", pc, False), ("varying consistency", post, True)]:
    res = ppc_share(p, vary)
    for j, t in enumerate((1.5, 2.0, 3.0)):
        lo, hi = np.percentile(res["replicated"][:, j], [5.5, 94.5])
        rows.append({"model": name, "threshold_T": t, "observed_share": res["observed"][:, j].mean(),
                     "replicated_mean": res["replicated"][:, j].mean(), "replicated_89%": f"[{lo:.3f}, {hi:.3f}]"})
pd.DataFrame(rows).round(3)

**Constant consistency understates the tails.** With every school equally consistent, the model expects 0.7% of schools to have $T_i > 3$ but 2.9% do, and 8.5% against 6.3% above 2. Letting consistency vary per school removes most of this: 6.8% above 2 (inside the replicated interval) and 1.1% above 3. So the data do contain schools whose elements scatter more than a common $\tau_e$ allows.

The fit is not perfect. The share above 3 (1.1%) is still at the top edge of the replicated interval [0.4%, 1.0%], and the differences at 1.5 are negligible in both models. The extra scatter is concentrated in a small number of schools, which is what a school-specific consistency is meant to capture. Whether it is *consistency* or unmodelled structure is what the residual check below examines.

## Which schools are outstanding, and consistently so?

Each school has a posterior for $g_i$ and $\log s_i$. The scatter shows the posterior means, coloured by how many elements the school has. The shared-scale summaries below tell us how well an individual school's values are pinned down: a posterior SD near the prior SD (1 for $g_i$, $\sigma_s$ for $\log s_i$) means the data taught us little about that school.

In [ ]:
g_post = post["g"].to_numpy().reshape(-1, n_schools)
log_s_post = post["log_s"].to_numpy().reshape(-1, n_schools)
g_mean, g_sd = g_post.mean(axis=0), g_post.std(axis=0)
ls_mean, ls_sd = log_s_post.mean(axis=0), log_s_post.std(axis=0)
n_el = counts.astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(g_mean, np.exp(ls_mean), c=n_el, cmap="viridis", s=8, alpha=0.6)
axes[0].axhline(1, color="grey", linewidth=0.8, linestyle="--")
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_yscale("log")
axes[0].set_xlabel("general GCSE quality, $g_i$ (posterior mean)")
axes[0].set_ylabel("consistency factor $s_i$ (posterior mean; lower = more consistent)")
plt.colorbar(sc, ax=axes[0], label="number of GCSE elements")
axes[1].scatter(g_mean, g_sd, s=6, alpha=0.4, color="#4C72B0")
axes[1].set_xlabel("$g_i$ (posterior mean)")
axes[1].set_ylabel("posterior SD of $g_i$")
axes[1].set_ylim(0, 1)
plt.show()

sigma_s_mean = post["sigma_s"].to_numpy().mean()
print(f"posterior SD of g_i: median {np.median(g_sd):.2f} (prior 1)")
print(f"posterior SD of log s_i: median {np.median(ls_sd):.2f} (prior {sigma_s_mean:.2f}, i.e. the posterior mean of sigma_s)")
print("correlation of posterior means, g vs log s:", np.corrcoef(g_mean, ls_mean)[0, 1].round(3))

**General quality is well determined; consistency is only partly.** For most schools (all six elements) the posterior SD of $g_i$ is about 0.19 on a prior scale of 1, so the ranking on general quality is reliable. The posterior SD of $\log s_i$ has median 0.28 against a prior of 0.35, so the data teach us relatively little about any one school's consistency. Treat an individual school's $s_i$ as a weak signal, not a ranking.

The schools with the most extreme values are almost all ones with three to five elements (dark points): low $g_i$, high $s_i$, and the widest posterior SDs in the right-hand panel (up to 0.7). Fewer elements give less information, so their values are pinned less firmly and also reflect a few very low or very scattered scores. Schools with six elements form a compact cloud with $s_i$ between about 0.7 and 1.8.

The tilt of the cloud (higher-quality schools slightly more consistent) matches $\rho = -0.19$. The correlation of the posterior *means* (-0.31) is stronger than $\rho$ because $g_i$ is well determined while the school-specific part of $\log s_i$ is shrunk hard, so the posterior means of $\log s_i$ lean more on $g_i$ than the true values do. $\rho$ is the number to quote.

## Is one general factor enough? A check on leftover structure

The raw correlations suggested the pairs Maths-Science and English-Open are slightly more alike than a single factor implies. If so, "consistency" as defined above would partly reflect a systematic split (for example maths-and-science schools versus language-and-open schools) instead of random scatter. We compute the standardised residuals $r_{ie}$ from the full model and their correlation across schools (averaged over posterior draws). Under a correct one-factor model these should be uncorrelated, apart from a small negative correlation created by estimating $g_i$ from the same data.

In [ ]:
mu_d, lam_d, tau_d = (draw_array(post, n) for n in ["mu", "lam", "tau"])
g_d, log_s_d = draw_array(post, "g"), draw_array(post, "log_s")
mean = mu_d[:, e_idx] + lam_d[:, e_idx] * g_d[:, s_idx]
r = (x_obs[None, :] - mean) / np.sqrt((tau_d[:, e_idx] * np.exp(log_s_d[:, s_idx])) ** 2 + x_se**2)

resid_wide = pd.DataFrame({"school": s_idx, "element": e_idx, "r": r.mean(axis=0)}).pivot(index="school", columns="element", values="r")
resid_wide.columns = elements
resid_corr = resid_wide.dropna().corr()

fig, ax = plt.subplots(figsize=(6.2, 5))
im = ax.imshow(resid_corr, vmin=-0.5, vmax=0.5, cmap="RdBu_r")
ax.set_xticks(range(6), elements, rotation=30)
ax.set_yticks(range(6), elements)
for i in range(6):
    for j in range(6):
        ax.text(j, i, f"{resid_corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.set_title("Correlation of standardised residuals")
ax.grid(False)
plt.colorbar(im, ax=ax)
plt.show()

**The residuals are correlated in a pattern, so one general factor is not quite enough.** Negative correlations are expected: $g_i$ is estimated from the same scores, which pulls residuals of the strongly loaded elements negative against each other (about -0.2 if six elements were equally informative). The English, Maths, Science and Open block is more negative than that (-0.22 to -0.44), and Languages, which carries little weight in $g_i$, is close to zero with everything.

What matters is the departure from that baseline. Maths-Science (+0.10) and English-Open (+0.08) are clearly above the rest of the block (about -0.35 on average), while the cross pairs (Maths-English, Maths-Open, Science-English, Science-Open) are the most negative. That is the same pairing seen in the raw correlations: a school's Maths and Science tend to be strong or weak together relative to its general level, and so do English and Open. The correlation is modest in size, but it shows some of the "element-specific" scatter is systematic, so part of what the model calls low consistency is a maths-and-science versus English-and-open split. A model with a second factor (or correlated $\delta_{ie}$) would separate the two; we have not fitted one.

## Regional pattern (descriptive)

The mean posterior general quality and consistency by region, with the number of schools. This is descriptive, not a separate model: the shrinkage in $g_i$ and $\log s_i$ already pulls poorly determined schools towards the overall average.

In [ ]:
region = pd.Series(urns.map(raw.drop_duplicates("URN").set_index("URN")["RGN24NM"]), index=range(n_schools))
by_region = pd.DataFrame({"g": g_mean, "s": np.exp(ls_mean), "region": region}).groupby("region").agg(
    schools=("g", "count"), mean_g=("g", "mean"), sd_g=("g", "std"), mean_s=("s", "mean")).sort_values("schools", ascending=False)
display(by_region.round(3))

order = list(by_region.index)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
data_g = [g_mean[(region == r_).to_numpy()] for r_ in order]
data_s = [np.exp(ls_mean[(region == r_).to_numpy()]) for r_ in order]
axes[0].boxplot(data_g, vert=False, tick_labels=order, showfliers=False)
axes[0].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("general quality $g_i$ (posterior mean)")
axes[1].boxplot(data_s, vert=False, tick_labels=order, showfliers=False)
axes[1].axvline(1, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("consistency factor $s_i$ (posterior mean)")
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

## Adding a regional layer

The baseline shrinks every school's general quality towards the *national* mean. But the descriptive table above shows regions differ (London +0.47, North East -0.45), so a poorly measured school in the North East is being pulled toward the wrong centre. We add region to the model in two places and ask what it changes:

$$
g_i \sim \text{Normal}(m_{r(i)},\ 1), \qquad \log s_i = c_{r(i)} + \sigma_s\left(\rho\,(g_i - m_{r(i)}) + \sqrt{1-\rho^2}\; w_i\right)
$$

- $m_r$ is the **regional mean of general quality**, partially pooled: $m_r \sim \text{Normal}(0, \sigma_m)$ constrained to sum to zero, so the element means $\mu_e$ stay identified. $\sigma_m$ is how much regions differ, in units of the within-region spread of schools (which is fixed at 1).
- $c_r$ is the **regional shift in mean log consistency**, also zero-sum, with spread $\sigma_c$. We expect this to be near zero, so it is mainly a check.
- $\rho$ is now the correlation *within* a region. Schools with no region (a handful) take the national average, $m = c = 0$.
- The measurement part of the model (loadings, specific SDs, known errors, marginalised $\delta_{ie}$) is unchanged.

Because the within-region spread is fixed at 1, the total SD of $g_i$ is now $\sqrt{1+\sigma_m^2}$, slightly above 1. To compare with the baseline we therefore express the loadings on the same scale by multiplying by that factor.

We can not use PSIS-LOO to compare the models (latent-variable models, see `NOTES.md`). Instead we ask four things: how big the regional effects are, whether any other parameter moves, which individual schools change, and whether the posterior predictive check improves.

In [ ]:
region_series = raw.drop_duplicates("URN").set_index("URN")["RGN24NM"].reindex(urns)
regions = list(region_series.value_counts().index)
reg_idx = region_series.map({r: k for k, r in enumerate(regions)}).fillna(len(regions)).astype(int).to_numpy()
print(f"{(reg_idx == len(regions)).sum()} schools without a region (given the national average)")

with pm.Model(coords={"element": elements, "region": regions}) as regional_model:
    mu = pm.Normal("mu", 0, 1, dims="element")
    lam = pm.HalfNormal("lam", 1, dims="element")
    tau = pm.HalfNormal("tau", 0.5, dims="element")
    sigma_m = pm.HalfNormal("sigma_m", 0.5)
    m = pm.ZeroSumNormal("m", sigma=sigma_m, dims="region")
    sigma_c = pm.HalfNormal("sigma_c", 0.2)
    c = pm.ZeroSumNormal("c", sigma=sigma_c, dims="region")
    m_all = pt.concatenate([m, pt.zeros(1)])   # last entry: schools with no region
    c_all = pt.concatenate([c, pt.zeros(1)])
    g = pm.Normal("g", m_all[reg_idx], 1, shape=n_schools)
    sigma_s = pm.HalfNormal("sigma_s", 0.5)
    rho = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
    w = pm.Normal("w", 0, 1, shape=n_schools)
    log_s = pm.Deterministic("log_s", c_all[reg_idx] + sigma_s * (rho * (g - m_all[reg_idx]) + pt.sqrt(1 - rho**2) * w))
    pm.Normal("x_obs", mu=mu[e_idx] + lam[e_idx] * g[s_idx],
              sigma=pt.sqrt((tau[e_idx] * pt.exp(log_s[s_idx]))**2 + x_se**2), observed=x_obs)

In [ ]:
with regional_model:
    idata_reg = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                          random_seed=RANDOM_SEED, progressbar=False)

### Diagnostics for the regional model

In [ ]:
print(f"regional model: divergences = {int(idata_reg.sample_stats['diverging'].sum())}")
core = az.summary(idata_reg, var_names=["mu", "lam", "tau", "sigma_m", "m", "sigma_c", "c", "sigma_s", "rho"], round_to=3)
print(f"worst r_hat = {core['r_hat'].max():.3f}, smallest bulk ESS = {core['ess_bulk'].min():.0f}, smallest tail ESS = {core['ess_tail'].min():.0f}")
core.loc[["sigma_m", "sigma_c", "sigma_s", "rho"]]

### How big are the regional effects?

Left: $m_r$, the regional shift in general quality (grey crosses show the mean of the *baseline* posterior $g_i$ by region, centred on the average of the nine regions so that it is on the same footing as the zero-sum $m_r$; the baseline $g_i$ has SD 1 overall, so its scale is slightly different). Right: $c_r$, the regional shift in log consistency. The share of variance in $g_i$ that is regional is $\sigma_m^2/(1+\sigma_m^2)$.

In [ ]:
post_r = idata_reg.posterior
m_draws = post_r["m"].to_numpy().reshape(-1, len(regions))
c_draws = post_r["c"].to_numpy().reshape(-1, len(regions))
sigma_m_d = post_r["sigma_m"].to_numpy().ravel()
sigma_c_d = post_r["sigma_c"].to_numpy().ravel()
print("sigma_m:", summarise(sigma_m_d))
print("sigma_c:", summarise(sigma_c_d))
print("regional share of variance in g:", summarise(sigma_m_d**2 / (1 + sigma_m_d**2)))

order_r = np.argsort(-m_draws.mean(axis=0))
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True)
for ax, draws, label, ref in [(axes[0], m_draws, "regional shift in general quality, $m_r$", 0.0),
                              (axes[1], c_draws, "regional shift in log consistency, $c_r$", 0.0)]:
    for row, k in enumerate(order_r):
        lo, med, hi = np.percentile(draws[:, k], [5.5, 50, 94.5])
        ax.plot([lo, hi], [row, row], color="#4C72B0", linewidth=2)
        ax.plot(med, row, "o", color="#4C72B0")
    ax.axvline(ref, color="grey", linewidth=0.8, linestyle="--")
    ax.set_xlabel(label)
axes[0].plot([by_region.loc[regions[k], "mean_g"] - by_region["mean_g"].mean() for k in order_r], range(len(regions)), "x", color="grey", label="baseline mean $g_i$ (centred)")
axes[0].legend(loc="lower right")
axes[0].set_yticks(range(len(regions)), [regions[k] for k in order_r])
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

table = pd.DataFrame({"schools": np.bincount(reg_idx)[:len(regions)],
                      "m_mean": m_draws.mean(axis=0), "m_lo": np.percentile(m_draws, 5.5, axis=0), "m_hi": np.percentile(m_draws, 94.5, axis=0),
                      "c_mean": c_draws.mean(axis=0), "c_lo": np.percentile(c_draws, 5.5, axis=0), "c_hi": np.percentile(c_draws, 94.5, axis=0)},
                     index=regions).iloc[order_r]
table.round(3)

### Does anything else in the model move?

We compare the parameters the two models share. The regional loadings are multiplied by $\sqrt{1+\sigma_m^2}$ so that both are on the scale of one SD of general quality across all schools.

In [ ]:
scale_r = np.sqrt(1 + sigma_m_d**2)
rows = []
for e in elements:
    lam_b = post["lam"].sel(element=e).to_numpy().ravel()
    lam_r = post_r["lam"].sel(element=e).to_numpy().ravel() * scale_r
    tau_b = post["tau"].sel(element=e).to_numpy().ravel()
    tau_r = post_r["tau"].sel(element=e).to_numpy().ravel()
    rows.append({"parameter": f"lam[{e}]", "baseline": lam_b.mean(), "regional": lam_r.mean(), "difference": lam_r.mean() - lam_b.mean(),
                 "regional 89%": f"[{np.percentile(lam_r, 5.5):.3f}, {np.percentile(lam_r, 94.5):.3f}]"})
    rows.append({"parameter": f"tau[{e}]", "baseline": tau_b.mean(), "regional": tau_r.mean(), "difference": tau_r.mean() - tau_b.mean(),
                 "regional 89%": f"[{np.percentile(tau_r, 5.5):.3f}, {np.percentile(tau_r, 94.5):.3f}]"})
for name in ["sigma_s", "rho"]:
    b, r_ = post[name].to_numpy().ravel(), post_r[name].to_numpy().ravel()
    rows.append({"parameter": name, "baseline": b.mean(), "regional": r_.mean(), "difference": r_.mean() - b.mean(),
                 "regional 89%": f"[{np.percentile(r_, 5.5):.3f}, {np.percentile(r_, 94.5):.3f}]"})
pd.DataFrame(rows).set_index("parameter").round(3)

### Which schools change?

The size of $g_i$ has no natural unit, and it is not pinned down on its own: the element means $\mu_e$ trade off against the overall mean of $g_i$, so $g_i$ can shift by a constant between the two models without meaning anything. So we compare the identified quantity, a school's **average expected value added across the six elements**, $\bar\mu + \bar\lambda\, g_i$ (in value-added points), which does not depend on that trade-off. The plots show how much it and each school's consistency move when the regional layer is added, against the number of elements the school has (fewer elements means less data, so more shrinkage towards the regional or national centre) and by region.

In [ ]:
g_post_r = post_r["g"].to_numpy().reshape(-1, n_schools)
log_s_post_r = post_r["log_s"].to_numpy().reshape(-1, n_schools)
mu_bar_b, lam_bar_b = post["mu"].to_numpy().mean(), post["lam"].to_numpy().mean()
mu_bar_r, lam_bar_r = post_r["mu"].to_numpy().mean(), post_r["lam"].to_numpy().mean()
G_b = mu_bar_b + lam_bar_b * g_mean
G_r = mu_bar_r + lam_bar_r * g_post_r.mean(axis=0)
ls_mean_r = log_s_post_r.mean(axis=0)
shift_G = G_r - G_b
shift_ls = ls_mean_r - ls_mean

region_arr = region_series.to_numpy()
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
jitter = np.random.default_rng(0).uniform(-0.15, 0.15, n_schools)
axes[0].scatter(n_el + jitter, shift_G, s=6, alpha=0.4, color="#4C72B0")
axes[0].axhline(0, color="grey", linewidth=0.8, linestyle="--")
axes[0].set_xlabel("number of GCSE elements")
axes[0].set_ylabel("change in average expected VA (VA points)")
axes[0].set_xticks([3, 4, 5, 6])
axes[1].boxplot([shift_G[region_arr == r_] for r_ in [regions[k] for k in order_r]], orientation="horizontal",
                tick_labels=[regions[k] for k in order_r], showfliers=False)
axes[1].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("change in average expected VA (VA points)")
axes[1].invert_yaxis()
axes[2].scatter(ls_mean, ls_mean_r, s=6, alpha=0.4, color="#DD8452")
axes[2].plot([-1, 1.5], [-1, 1.5], color="grey", linewidth=0.8, linestyle="--")
axes[2].set_xlabel("baseline posterior mean of $\\log s_i$")
axes[2].set_ylabel("regional-model posterior mean of $\\log s_i$")
plt.tight_layout()
plt.show()

full6 = n_el == 6
print(f"correlation of average expected VA, baseline vs regional: {np.corrcoef(G_b, G_r)[0, 1]:.4f}")
print(f"correlation of log s, baseline vs regional: {np.corrcoef(ls_mean, ls_mean_r)[0, 1]:.4f}")
print(f"mean change in average expected VA (all schools): {shift_G.mean():.4f} VA points")
print(f"mean |change| in average expected VA: six elements {np.abs(shift_G[full6]).mean():.4f}, fewer than six {np.abs(shift_G[~full6]).mean():.4f} VA points")
print(f"largest |change| in average expected VA: {np.abs(shift_G).max():.3f} VA points")
print(f"mean |change| in log s: six elements {np.abs(shift_ls[full6]).mean():.4f}, fewer than six {np.abs(shift_ls[~full6]).mean():.4f}")

### Does the posterior predictive check improve?

The same check as before, now for all three models.

In [ ]:
rows = []
for name, p_, vary in [("constant consistency", pc, False), ("varying consistency", post, True), ("varying + regional", post_r, True)]:
    res = ppc_share(p_, vary)
    for j, t in enumerate((1.5, 2.0, 3.0)):
        lo, hi = np.percentile(res["replicated"][:, j], [5.5, 94.5])
        rows.append({"model": name, "threshold_T": t, "observed_share": res["observed"][:, j].mean(),
                     "replicated_mean": res["replicated"][:, j].mean(), "replicated_89%": f"[{lo:.3f}, {hi:.3f}]"})
pd.DataFrame(rows).round(3)

**Regions differ in general quality, but adding them changes almost nothing else.**

- **Quality differs by region.** $\sigma_m = 0.32$ [0.20, 0.49] (in units of the within-region spread of schools), so about 10% of the variance in $g_i$ [4%, 19%] is regional, in line with the roughly 9% seen in the raw data. London is clearly high ($m_r = +0.59$) and the North East clearly low ($-0.36$), a gap of about 0.95 within-region SDs, or about 0.5 value-added points on an average element. The South East is modestly above ($+0.16$); the West Midlands, East Midlands and North West are below, with intervals that exclude zero; the South West, East of England and Yorkshire are not distinguishable from zero.
- **The baseline already had the regional means right.** The grey crosses (mean of the baseline $g_i$ by region) sit almost exactly on the regional estimates. With 70-390 schools per region and $g_i$ determined to about 0.19, the data outweigh the shrinkage, so shrinking towards the region instead of the nation moves little.
- **Consistency barely varies by region.** $\sigma_c = 0.07$ [0.03, 0.12]: regional shifts are at most about 8% in element-specific scatter, against 35% between schools within a region. Three of nine intervals exclude zero (London and Yorkshire more scattered, East of England less), but with nine regions and 89% intervals, and effects this small, we would not read anything into these.
- **Nothing else moves.** Loadings, $\tau_e$ and $\sigma_s$ agree with the baseline to within 0.006 (the regional loadings are rescaled by $\sqrt{1+\sigma_m^2}$ to the same units); $\rho$ goes from $-0.19$ to $-0.21$, less than one posterior SD. The posterior predictive check is identical for the two models.
- **School-level results barely change.** A school's average expected value added moves by 0.005 points on average for schools with six elements and 0.026 for those with fewer, the largest change being 0.19 (a three-element school). The regional effect on a school is a shift of about $\pm 0.01$ points by region (London up, North East down), as the shrinkage target moves. The correlation with the baseline is 0.9997 for quality and 0.987 for $\log s_i$.

So the regional layer is well supported for $g_i$ and unsupported for consistency, and it does not change the school-level conclusions. We would keep it for the link to A-level, where separating a regional shift in GCSE quality from a separate regional shift at A-level is the whole point, and because it gives regional estimates with uncertainty. For the GCSE results alone the baseline is enough.

## Sensitivity: how much does the result depend on Open?

The Open element is a broad bucket. It holds each pupil's best remaining GCSEs beyond English, Maths and the EBacc buckets, which for many pupils means arts subjects such as music, art and drama (the exact rules are in the DfE technical guide). If schools that do well across the board also do well in these, Open carries something the academic elements do not, and the model could depend on it. The residual check showed English-Open and Maths-Science pairing more strongly than one factor allows, so Open also plays a part in that structure.

We refit the full model **without Open** (five elements) and compare it with the six-element fit on the same schools. We ask whether general quality and the consistency ranking change, and whether $\sigma_s$ and $\rho$ survive. Then a descriptive check: is the extra scatter of the least consistent schools concentrated in Open, or spread over all elements?

In [ ]:
els5 = [e for e in elements if e != "Open"]
sub = long[long["element"].isin(els5)]
n_sub = sub.groupby("URN")["element"].nunique()
sub = sub[sub["URN"].isin(n_sub[n_sub >= 3].index)]   # schools with only three elements lose one and drop out
urns5 = pd.Index(sorted(sub["URN"].unique()))
sub_school = urns5.get_indexer(sub["URN"])
sub_elem = sub["element"].map({e: k for k, e in enumerate(els5)}).to_numpy()
sub_x, sub_se = sub["va"].to_numpy(), sub["se"].to_numpy()
n5 = len(urns5)
print(f"{n5} schools remain of {n_schools} ({n_schools - n5} had too few elements without Open)")

with pm.Model(coords={"element": els5}) as no_open_model:
    mu5 = pm.Normal("mu", 0, 1, dims="element")
    lam5 = pm.HalfNormal("lam", 1, dims="element")
    tau5 = pm.HalfNormal("tau", 0.5, dims="element")
    g5 = pm.Normal("g", 0, 1, shape=n5)
    sigma_s5 = pm.HalfNormal("sigma_s", 0.5)
    rho5 = pm.Deterministic("rho", 2 * pm.Beta("rho_raw", 2, 2) - 1)
    w5 = pm.Normal("w", 0, 1, shape=n5)
    log_s5 = pm.Deterministic("log_s", sigma_s5 * (rho5 * g5 + pt.sqrt(1 - rho5**2) * w5))
    pm.Normal("x_obs", mu=mu5[sub_elem] + lam5[sub_elem] * g5[sub_school],
              sigma=pt.sqrt((tau5[sub_elem] * pt.exp(log_s5[sub_school]))**2 + sub_se**2), observed=sub_x)
    idata_no_open = pm.sample(draws=1000, tune=2000, chains=4, target_accept=0.99,
                              random_seed=RANDOM_SEED, progressbar=False)

In [ ]:
post_n = idata_no_open.posterior
print(f"no-Open model: divergences = {int(idata_no_open.sample_stats['diverging'].sum())}")
core_n = az.summary(idata_no_open, var_names=["mu", "lam", "tau", "sigma_s", "rho"], round_to=3)
print(f"worst r_hat = {core_n['r_hat'].max():.3f}, smallest bulk ESS = {core_n['ess_bulk'].min():.0f}")
core_n.loc[["sigma_s", "rho"]]

In [ ]:
from scipy.stats import spearmanr
pos = urns.get_indexer(urns5)                       # the same schools in the six-element fit
g6, g5m = g_mean[pos], post_n["g"].to_numpy().reshape(-1, n5).mean(axis=0)
ls6, ls5m = ls_mean[pos], post_n["log_s"].to_numpy().reshape(-1, n5).mean(axis=0)
g5_sd = post_n["g"].to_numpy().reshape(-1, n5).std(axis=0)

k = int(0.1 * n5)
def overlap(x, y, largest):
    pick = (lambda v: set(np.argsort(-v)[:k])) if largest else (lambda v: set(np.argsort(v)[:k]))
    return len(pick(x) & pick(y)) / k
z = np.abs(g6 - g5m) / np.sqrt(g_sd[pos]**2 + g5_sd**2)

print(f"general quality g:   correlation {np.corrcoef(g6, g5m)[0, 1]:.3f}, rank correlation {spearmanr(g6, g5m)[0]:.3f}")
print(f"  top tenth overlap {overlap(g6, g5m, True):.2f}, bottom tenth overlap {overlap(g6, g5m, False):.2f}; "
      f"share of schools whose g moves by more than 1 / 2 combined SDs: {(z > 1).mean():.3f} / {(z > 2).mean():.3f}")
print(f"log consistency:     correlation {np.corrcoef(ls6, ls5m)[0, 1]:.3f}, rank correlation {spearmanr(ls6, ls5m)[0]:.3f}")
print(f"  most consistent tenth overlap {overlap(ls6, ls5m, False):.2f}, least consistent tenth overlap {overlap(ls6, ls5m, True):.2f} (0.10 would be chance)")
print("sigma_s: six elements", summarise(post["sigma_s"].to_numpy().ravel()), "\n         no Open     ", summarise(post_n["sigma_s"].to_numpy().ravel()))
print("rho:     six elements", summarise(post["rho"].to_numpy().ravel()), "\n         no Open     ", summarise(post_n["rho"].to_numpy().ravel()))

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
axes[0].scatter(g6, g5m, s=5, alpha=0.4, color="#4C72B0")
axes[0].set_xlabel("$g_i$, six elements"); axes[0].set_ylabel("$g_i$, without Open")
axes[1].scatter(ls6, ls5m, s=5, alpha=0.4, color="#DD8452")
axes[1].set_xlabel("$\\log s_i$, six elements"); axes[1].set_ylabel("$\\log s_i$, without Open")
for ax, lim in [(axes[0], (-5, 5)), (axes[1], (-1.2, 1.6))]:
    ax.plot(lim, lim, color="grey", linewidth=0.8, linestyle="--")
for ax, name, label in [(axes[2], "sigma_s", "$\\sigma_s$"), (axes[3], "rho", "$\\rho$")]:
    ax.hist(post[name].to_numpy().ravel(), bins=40, density=True, alpha=0.6, color="#4C72B0", label="six elements")
    ax.hist(post_n[name].to_numpy().ravel(), bins=40, density=True, alpha=0.6, color="#DD8452", label="without Open")
    ax.set_xlabel(label)
axes[3].axvline(0, color="grey", linewidth=0.8, linestyle="--")
axes[2].legend()
plt.tight_layout()
plt.show()

comp = pd.DataFrame({"loading six": post["lam"].mean(("chain", "draw")).to_series(), "loading no Open": post_n["lam"].mean(("chain", "draw")).to_series(),
                     "tau six": post["tau"].mean(("chain", "draw")).to_series(), "tau no Open": post_n["tau"].mean(("chain", "draw")).to_series()}).loc[elements]
comp.round(3)

### Is the extra scatter concentrated in Open?

In the model, consistency multiplies every element's specific scatter by the same factor $s_i$. If the least consistent schools were uneven mainly *because of Open*, Open would carry a larger share of their specific variance than it does across schools generally. For each element we estimate the specific variance (squared departure from $\mu_e + \lambda_e g_i$, minus the known sampling variance) among the least consistent tenth and among the most consistent tenth of schools, and compare each element's *share* of the total. Languages has a much larger specific variance than the others and dominates the totals, so we also show the shares among the other five elements. This is descriptive, because the schools are picked by $s_i$, which was itself estimated from the same scores; the comparison *across elements* is what is informative. (Ratios of the two groups are not shown: in the most consistent tenth the estimated specific variance is about zero, so they are unstable.)

In [ ]:
mu_m = post["mu"].to_numpy().reshape(-1, n_elements).mean(axis=0)
lam_m = post["lam"].to_numpy().reshape(-1, n_elements).mean(axis=0)
excess = (x_obs - mu_m[e_idx] - lam_m[e_idx] * g_mean[s_idx])**2 - x_se**2     # estimated specific variance per observation
lo, hi = np.quantile(ls_mean, [0.1, 0.9])
least, most = (ls_mean[s_idx] >= hi), (ls_mean[s_idx] <= lo)
spec = pd.DataFrame({e: {"least consistent tenth": excess[(e_idx == k_) & least].mean(),
                         "most consistent tenth": excess[(e_idx == k_) & most].mean(),
                         "all schools": excess[e_idx == k_].mean()} for k_, e in enumerate(elements)}).T
share = spec[["least consistent tenth", "all schools"]] / spec[["least consistent tenth", "all schools"]].sum()
no_lang = spec.drop(index="Languages")[["least consistent tenth", "all schools"]]
share_no_lang = no_lang / no_lang.sum()
out = spec.copy()
out["share, least tenth"] = share["least consistent tenth"]
out["share, all schools"] = share["all schools"]
out["share excl. Languages, least tenth"] = share_no_lang["least consistent tenth"]
out["share excl. Languages, all schools"] = share_no_lang["all schools"]
out.round(3)

**General quality does not depend on Open; consistency does, and the link between the two does.**

- **General quality is robust.** Without Open, $g_i$ correlates 0.988 with the six-element version, 90% of the top tenth and 86% of the bottom tenth stay where they were, and no school's $g_i$ moves by more than two combined SDs. Who is a high-quality school does not hinge on Open.
- **Consistency is less robust.** The rank correlation of $\log s_i$ falls to 0.88 and only 66% of the most consistent tenth stays in it (chance would be 10%). Schools still clearly differ in consistency without Open ($\sigma_s$ 0.36 falls to 0.31, both far from zero), so consistency is not just an Open effect, but a sizeable part of the ranking is.
- **$\rho$ does depend on it.** With six elements $\rho = -0.19$ [-0.25, -0.12]; without Open it is $-0.03$ [-0.10, 0.05], i.e. no detectable link. The finding that higher-quality schools are slightly more consistent is carried by Open, and we should treat it as tentative.
- **Open shapes what $g_i$ means.** Without it English's specific SD rises (0.175 to 0.215) while Maths's and Science's fall (0.173 to 0.152, 0.166 to 0.129): $g_i$ tilts towards the Maths-Science pair. Dropping Open does not remove the pairing seen in the residual check; it changes which pair defines the general factor.
- **Where the extra scatter sits.** Among the five elements other than Languages, Open accounts for about a third of the specific variance in the least consistent tenth of schools (35%) against 27% across schools, English for 29% against 24%, and Maths, Science and Humanities for less than their usual share. So the least consistent schools depart from their general level more than usual on Open and English, and less than usual on the rest. It is a tilt, not a concentration.

**Interpretation, which is the author's reading, not something the data can settle.** Open is the only element in which arts subjects such as music, art and drama count, so a school's strength or weakness there against its academic profile is where breadth and "roundedness" would show up. That fits the results above: Open matters to the consistency estimate and to its link with quality. But Open is a single score for a bucket whose contents differ from school to school (arts, other approved qualifications, and overflow from the other buckets), so we cannot tell arts from anything else in it. Testing the idea would need entries and results in the individual arts subjects, which are not in this data.

## Summary

- **One general GCSE quality explains most of the variation.** Each of English, Maths, Science, Humanities and Open loads on $g_i$ at 0.46-0.58 per SD, with $\tau_e$ of 0.17-0.21 and 82-87% of their between-school variance being general. Languages is the exception: loading 0.49 but $\tau_e = 0.60$, only 37% general and 53% element-specific.
- **Schools do differ in consistency.** $\sigma_s = 0.36$ [0.33, 0.38], so a school one SD more scattered than average has element-specific scatter about 1.4 times typical, and one SD more consistent about 0.7 times. Allowing this fixes the excess of very poorly fitting schools that the constant-consistency model leaves.
- **Higher quality goes weakly with higher consistency, but only when Open is included.** $\rho = -0.19$ [-0.25, -0.12] with all six elements; without Open it is $-0.03$ [-0.10, 0.05]. Treat it as tentative.
- **Regions differ in quality, not in consistency.** With a regional layer, about 10% of the variance in $g_i$ is regional ($\sigma_m = 0.32$): London is highest ($m_r = +0.59$) and the North East lowest ($-0.36$), a gap of about 0.95 within-region SDs. Regional shifts in consistency are at most about 8% ($\sigma_c = 0.07$). Adding the regional layer changes none of the other parameters and moves no school's values by more than 0.19 value-added points (median about 0.005).

### Cautions

- **Consistency depends on Open.** General quality is robust to leaving Open out (correlation 0.988), but the consistency ranking is not (rank correlation 0.88, and 66% of the most consistent tenth stays). Open is a mixed bucket, so we cannot say which of its subjects matter.
- **Consistency is only partly identified per school** (posterior SD of $\log s_i$ 0.28 against 0.35). Schools with fewer than six elements are the least certain.
- **The consistency factor also absorbs systematic structure.** The residual check shows Maths-Science and English-Open pairing, and the posterior predictive check leaves a small excess in the extreme tail. Some "inconsistency" is therefore a subject-group split, not random scatter.
- **Sampling is adequate but not tidy for $\mu_e$**: $\hat R$ is 1.02 and ESS about 230 for the $\mu_e$ (no divergences; other parameters are fine). $\mu_e$ and the mean of $g_i$ trade off. This does not affect the interpretation above, but a stricter fit would centre $g_i$ or use a longer run.
- **Single year**, and these are associations, not effects of teaching. The regional model has no region-by-element deviations.

### Next steps

1. Decide whether to add a second factor (or correlated element departures) for the Maths-Science / English-Open structure, then repeat the drop-Open check to see whether $\rho$ survives.
2. Link $g_i$ and $s_i$ (with the regional layer) to A-level value added, ideally jointly so that their uncertainty carries through.